# Modul 3: Motion Tracking — odtwarzanie ruchu na G1

W tym notebooku przejdziemy przez **pipeline BeyondMimic** — od zapisu ruchu czlowieka, przez retargeting na robota humanoidalnego **Unitree G1**, az po odtworzenie trajektorii w symulatorze MuJoCo.

**Pipeline BeyondMimic** sklada sie z nastepujacych krokow:
1. **Motion Capture** — nagranie ruchu czlowieka (np. dataset LAFAN1)
2. **Retargeting** — przeniesienie ruchu na kinematyke robota G1 (23 DOF)
3. **Wizualizacja** — odtworzenie retargetowanego ruchu w MuJoCo
4. **Trening RL** — nauczenie polityki odtwarzajacej ruch z zachowaniem rownowagi

W tym cwiczeniu skupimy sie na krokach 2-3: ustawianiu poz robota, generowaniu trajektorii i wizualizacji w MuJoCo.

In [ ]:
!pip install mujoco mujoco_menagerie numpy matplotlib -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import mujoco
import mujoco_menagerie
import os

# Zaladuj model G1 z mujoco_menagerie
menagerie_path = os.path.dirname(mujoco_menagerie.__file__)
model_path = os.path.join(menagerie_path, "unitree_g1", "g1.xml")
print(f"Model path: {model_path}")

model = mujoco.MjModel.from_xml_path(model_path)
data = mujoco.MjData(model)

print(f"Liczba stopni swobody (nq): {model.nq}")
print(f"Liczba aktuatorow (nu): {model.nu}")
print(f"Liczba cial (nbody): {model.nbody}")

# Funkcja renderowania
renderer = mujoco.Renderer(model, height=480, width=640)

def render_frame(data):
    """Renderuj pojedyncza klatke z aktualnego stanu symulacji."""
    mujoco.mj_forward(model, data)
    renderer.update_scene(data)
    return renderer.render()

# Test — renderuj pozycje startowa
mujoco.mj_resetData(model, data)
img = render_frame(data)
plt.figure(figsize=(8, 6))
plt.imshow(img)
plt.axis('off')
plt.title('G1 — pozycja domyslna')
plt.show()
print("Model G1 zaladowany pomyslnie!")

## Kinematyka robota — ustawianie poz

Robot G1 ma **free joint** (7 wartosci: 3 pozycja + 4 kwaternion) oraz stawy obrotowe.
Wektor `data.qpos` zawiera pelny stan konfiguracji:
- `qpos[0:3]` — pozycja bazy (x, y, z)
- `qpos[3:7]` — orientacja bazy (kwaternion w, x, y, z)
- `qpos[7:]` — katy stawow (kolana, biodra, kostki, ramiona, itp.)

Ponizej ustawiamy robota w roznych pozach, zmieniajac wartosci stawow.

In [ ]:
# Definiujemy 4 rozne pozy robota
# Indeksy stawow zaleza od modelu — uzyjemy ogolnych offsetow

def set_pose_standing(data):
    """Pozycja stojaca — domyslna (keyframe)."""
    mujoco.mj_resetData(model, data)
    # Ustaw robota nad ziemia
    data.qpos[2] = 0.75  # wysokosc
    mujoco.mj_forward(model, data)

def set_pose_crouch(data):
    """Przysiad — zgiete kolana."""
    mujoco.mj_resetData(model, data)
    data.qpos[2] = 0.55  # nizej
    nq_free = 7  # free joint
    joint_angles = np.zeros(model.nq - nq_free)
    # Zginamy biodra i kolana (przyblizony przysiad)
    # Biodra pitch (lewe, prawe) — zginamy do przodu
    joint_angles[0] = -0.6   # left_hip_pitch
    joint_angles[5] = -0.6   # right_hip_pitch
    # Kolana — zginamy
    joint_angles[2] = 1.2    # left_knee
    joint_angles[7] = 1.2    # right_knee
    # Kostki — kompensacja
    joint_angles[3] = -0.6   # left_ankle_pitch
    joint_angles[8] = -0.6   # right_ankle_pitch
    data.qpos[nq_free:] = joint_angles[:model.nq - nq_free]
    mujoco.mj_forward(model, data)

def set_pose_arms_raised(data):
    """Rece podniesione do gory."""
    mujoco.mj_resetData(model, data)
    data.qpos[2] = 0.75
    nq_free = 7
    joint_angles = np.zeros(model.nq - nq_free)
    # Ramiona — podnosimy (shoulder pitch)
    n_leg_joints = 10  # przyblizenie: 5 stawow na noge
    if model.nq - nq_free > n_leg_joints + 4:
        joint_angles[n_leg_joints] = -2.5     # left_shoulder_pitch (do gory)
        joint_angles[n_leg_joints + 3] = -2.5  # right_shoulder_pitch
    data.qpos[nq_free:] = joint_angles[:model.nq - nq_free]
    mujoco.mj_forward(model, data)

def set_pose_walking(data):
    """Pozycja w srodku kroku — asymetryczne nogi."""
    mujoco.mj_resetData(model, data)
    data.qpos[2] = 0.72
    nq_free = 7
    joint_angles = np.zeros(model.nq - nq_free)
    # Lewa noga do przodu, prawa do tylu
    joint_angles[0] = -0.4   # left_hip_pitch (do przodu)
    joint_angles[2] = 0.3    # left_knee (lekko zgiete)
    joint_angles[3] = -0.15  # left_ankle
    joint_angles[5] = 0.3    # right_hip_pitch (do tylu)
    joint_angles[7] = 0.6    # right_knee (zgiete)
    joint_angles[8] = -0.3   # right_ankle
    data.qpos[nq_free:] = joint_angles[:model.nq - nq_free]
    mujoco.mj_forward(model, data)

# Renderuj 4 pozy jako subplot
poses = [
    ("Stojaca (keyframe)", set_pose_standing),
    ("Przysiad (zgiete kolana)", set_pose_crouch),
    ("Rece podniesione", set_pose_arms_raised),
    ("Srodek kroku (chod)", set_pose_walking),
]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, (title, pose_fn) in zip(axes, poses):
    pose_fn(data)
    img = render_frame(data)
    ax.imshow(img)
    ax.set_title(title, fontsize=12)
    ax.axis('off')

plt.suptitle('Pozy robota G1', fontsize=16)
plt.tight_layout()
plt.show()

## Generowanie trajektorii ruchu

W prawdziwym pipeline BeyondMimic trajektorie pochodza z motion capture (np. LAFAN1).
Tutaj wygenerujemy **uproszczona trajektorie chodu** opartego na ruchach sinusoidalnych — kazdy staw nogi wykonuje oscylacje z odpowiednim przesuniecem fazowym.

In [ ]:
# Generowanie trajektorii chodu (sinusoidy)
dt = 0.02  # 50 Hz
duration = 2.4  # sekundy
num_frames = int(duration / dt)  # 120 klatek
t = np.linspace(0, duration, num_frames)

nq_free = 7
nq_joints = model.nq - nq_free

# Trajektoria stawow: (num_frames, nq_joints)
trajectory = np.zeros((num_frames, nq_joints))

# Parametry chodu
freq = 1.5  # Hz — czestotliwosc kroku
omega = 2 * np.pi * freq

for i in range(num_frames):
    phase = omega * t[i]
    
    # Lewa noga
    trajectory[i, 0] = -0.3 * np.sin(phase)          # left_hip_pitch
    trajectory[i, 2] = 0.4 * (1 - np.cos(phase)) / 2  # left_knee (zawsze >= 0)
    trajectory[i, 3] = -0.15 * np.sin(phase)          # left_ankle_pitch
    
    # Prawa noga (przesuniecie fazowe o pi)
    trajectory[i, 5] = -0.3 * np.sin(phase + np.pi)   # right_hip_pitch
    trajectory[i, 7] = 0.4 * (1 - np.cos(phase + np.pi)) / 2  # right_knee
    trajectory[i, 8] = -0.15 * np.sin(phase + np.pi)  # right_ankle_pitch

print(f"Wygenerowano trajektorie: {trajectory.shape[0]} klatek x {trajectory.shape[1]} stawow")
print(f"Czas trwania: {duration}s, dt: {dt}s, czestotliwosc kroku: {freq} Hz")

In [ ]:
# Animacja trajektorii — renderujemy co 10. klatke (4x3 = 12 klatek)
frame_indices = np.arange(0, num_frames, max(1, num_frames // 12))[:12]

fig, axes = plt.subplots(3, 4, figsize=(20, 15))
axes_flat = axes.flatten()

for idx, frame_i in enumerate(frame_indices):
    mujoco.mj_resetData(model, data)
    data.qpos[2] = 0.75  # wysokosc bazy
    data.qpos[nq_free:] = trajectory[frame_i, :nq_joints]
    mujoco.mj_forward(model, data)
    
    img = render_frame(data)
    axes_flat[idx].imshow(img)
    axes_flat[idx].set_title(f'Klatka {frame_i} (t={frame_i*dt:.2f}s)', fontsize=10)
    axes_flat[idx].axis('off')

plt.suptitle('Trajektoria chodu — co 10. klatka', fontsize=16)
plt.tight_layout()
plt.show()

## Analiza trajektorii

Wizualizujemy katy stawow w czasie, aby zrozumiec wzorce ruchu.
Prawidlowy chod powinien miec:
- Przeciwfazowe biodra (lewe/prawe)
- Kolana zawsze zgiete w fazie wymachu
- Kostki kompensujace pochylenie stopy

In [ ]:
# Wykresy katow stawow w czasie
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Hip pitch
axes[0].plot(t, np.degrees(trajectory[:, 0]), 'b-', linewidth=2, label='Lewe biodro')
axes[0].plot(t, np.degrees(trajectory[:, 5]), 'r--', linewidth=2, label='Prawe biodro')
axes[0].set_ylabel('Kat [stopnie]')
axes[0].set_title('Hip Pitch (biodra)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=0, color='k', linewidth=0.5)

# Knee
axes[1].plot(t, np.degrees(trajectory[:, 2]), 'b-', linewidth=2, label='Lewe kolano')
axes[1].plot(t, np.degrees(trajectory[:, 7]), 'r--', linewidth=2, label='Prawe kolano')
axes[1].set_ylabel('Kat [stopnie]')
axes[1].set_title('Knee (kolana)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Ankle
axes[2].plot(t, np.degrees(trajectory[:, 3]), 'b-', linewidth=2, label='Lewa kostka')
axes[2].plot(t, np.degrees(trajectory[:, 8]), 'r--', linewidth=2, label='Prawa kostka')
axes[2].set_ylabel('Kat [stopnie]')
axes[2].set_xlabel('Czas [s]')
axes[2].set_title('Ankle Pitch (kostki)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)
axes[2].axhline(y=0, color='k', linewidth=0.5)

plt.suptitle('Katy stawow w czasie — trajektoria chodu', fontsize=14)
plt.tight_layout()
plt.show()

print("Obserwacje:")
print("- Biodra sa w przeciwfazie (lewe/prawe przesuniete o pi)")
print("- Kolana zginaja sie w fazie wymachu nogi")
print("- Kostki kompensuja pochylenie stopy przy kontakcie z podlozem")

## Export do symulatora przegladarkowego

Eksportujemy wygenerowana trajektorie do formatu JSON, ktory mozna wgrac do przegladarkowego symulatora G1 (zakladka **Trajektoria**).

In [ ]:
import json

# Przygotuj klatki do eksportu
# Kazda klatka = pelny qpos (free joint + stawy)
frames_list = []
for i in range(num_frames):
    frame_qpos = np.zeros(model.nq)
    frame_qpos[2] = 0.75           # wysokosc
    frame_qpos[3] = 1.0            # kwaternion w=1 (brak rotacji)
    frame_qpos[nq_free:] = trajectory[i, :nq_joints]
    frames_list.append(frame_qpos.tolist())

trajectory_export = {
    "name": "Walking demo",
    "dt": 0.02,
    "nq": model.nq,
    "frames": frames_list
}

with open("trajectory.json", "w") as f:
    json.dump(trajectory_export, f)

print(f"Eksportowano trajectory.json — wgraj do zakladki Trajektoria w symulatorze")
print(f"  Liczba klatek: {len(frames_list)}")
print(f"  nq: {model.nq}")
print(f"  dt: {trajectory_export['dt']}s")
print(f"  Czas trwania: {len(frames_list) * trajectory_export['dt']:.1f}s")

## Pipeline BeyondMimic

Pelny pipeline treningowy **BeyondMimic** dla robota humanoidalnego G1 sklada sie z nastepujacych etapow:

### 1. Pozyskanie danych ruchu (Motion Capture)
- Datasety: **LAFAN1**, **AMASS**, **CMU MoCap**
- Format: pozycje 3D markerow lub katy stawow ludzkiego szkieletu
- Czestotliwosc: 30-120 Hz

### 2. Retargeting na robota
- Mapowanie kinematyki ludzkiej na kinematyke G1 (23 DOF)
- Rozwiazanie kinematyki odwrotnej (IK) dla kazdej klatki
- Weryfikacja limitow stawow i unikanie kolizji

### 3. Trening polityki RL (Reinforcement Learning)
- **Srodowisko**: MuJoCo z modelem G1
- **Obserwacja**: aktualne katy stawow + docelowe katy z trajektorii + czujniki IMU
- **Akcja**: momenty sil na stawach
- **Nagroda**: bliskosc do trajektorii referencyjnej + kary za upadek
- **Algorytm**: PPO (Proximal Policy Optimization) — omowiony w Module 2

### 4. Sim-to-Real Transfer
- **Domain Randomization**: losowe zmiany masy, tarcia, opoznien
- **Wdrozenie na robota**: polityka dziala na Jetson Orin w czasie rzeczywistym (500 Hz)

### Schemat pipeline
```
Motion Capture (LAFAN1)
       |
       v
Retargeting (IK) --> Trajektoria referencyjna (qpos)
       |                        |
       v                        v
  MuJoCo Env  <----  Nagroda: ||q - q_ref||
       |                        
       v                        
  PPO Training --> Polityka (neural network)
       |                        
       v                        
  Sim-to-Real --> Wdrozenie na G1
```

W nastepnych modulach (4-6) zintegrujemy te polityki z percepcja i nawigacja, tworzy pelny system autonomiczny.